In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import tkinter as tk
from tkinter import filedialog, messagebox
from pathlib import Path

def select_file(title, file_types, save=False):
    """Unified file selection function"""
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    try:
        if save:
            file_path = filedialog.asksaveasfilename(
                title=title,
                filetypes=file_types,
                defaultextension=file_types[0][1]
            )
        else:
            file_path = filedialog.askopenfilename(
                title=title,
                filetypes=file_types
            )
    finally:
        root.destroy()
    
    return file_path if file_path else None

def optimize_dataframe(df):
    """Optimize DataFrame memory usage"""
    date_columns = ['DOB', 'TRAVEL DATE', 'RETURN DATE']
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
    
    categorical_columns = ['GENDER', 'MARITAL STATUS', 'NATIONALITY', 
                         'PURPOSE OF VISIT', 'CARRIER TYPE', 'CARRIER NAME',
                         'EMBARK PORT']
    for col in categorical_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    
    return df

def calculate_age_vectorized(dob_series):
    """Calculate age using vectorized operations"""
    today = pd.Timestamp.now()
    return ((today - dob_series).dt.days / 365.25)

def create_table1(df):
    """Create Table 1 with interactions"""
    results = []
    total_n = len(df)
    
    # Calculate age
    print("Calculating age statistics...")
    df['AGE'] = calculate_age_vectorized(df['DOB'])
    age_stats = df['AGE'].agg(['mean', 'std'])
    results.append({
        'Category': 'Demographics',
        'Characteristic': 'Age',
        'Value': f"{age_stats['mean']:.1f} ± {age_stats['std']:.1f}",
        'N': total_n
    })
    
    # Process categorical variables
    categorical_vars = {
        'Demographics': ['GENDER', 'MARITAL STATUS', 'NATIONALITY'],
        'Travel': ['PURPOSE OF VISIT', 'CARRIER TYPE', 'CARRIER NAME', 'EMBARK PORT']
    }
    
    print("Processing categorical variables...")
    for category, variables in categorical_vars.items():
        for var in variables:
            value_counts = df[var].value_counts().head(5)  # Top 5 for all categories
            percentages = (value_counts / total_n * 100)
            
            results.append({
                'Category': category,
                'Characteristic': f"{var.title().replace('_', ' ')} (Top 5)",
                'Value': '',
                'N': '',
                'Percentage': ''
            })
            
            for val, count in value_counts.items():
                results.append({
                    'Category': category,
                    'Characteristic': "   " + str(val),
                    'Value': '',
                    'N': count,
                    'Percentage': f"{percentages[val]:.1f}%"
                })
    
    # Create and analyze Gender-Marital Status combinations
    print("Analyzing Gender-Marital Status combinations...")
    df['GENDER_MARITAL'] = df['GENDER'].astype(str) + ' - ' + df['MARITAL STATUS'].astype(str)
    gender_marital_counts = df['GENDER_MARITAL'].value_counts()
    gender_marital_pct = (gender_marital_counts / total_n * 100)
    
    results.append({
        'Category': 'Demographics',
        'Characteristic': 'Gender-Marital Status Combinations (Top 5)',
        'Value': '',
        'N': '',
        'Percentage': ''
    })
    
    for val, count in gender_marital_counts.head(5).items():
        results.append({
            'Category': 'Demographics',
            'Characteristic': "   " + val,
            'Value': '',
            'N': count,
            'Percentage': f"{gender_marital_pct[val]:.1f}%"
        })
    
    # Calculate length of stay and its interaction with purpose of visit
    print("Calculating length of stay statistics...")
    df['LENGTH_OF_STAY'] = (df['RETURN DATE'] - df['TRAVEL DATE']).dt.days
    
    # Overall length of stay
    stay_stats = df['LENGTH_OF_STAY'].agg(['mean', 'std'])
    results.append({
        'Category': 'Travel',
        'Characteristic': 'Length of Stay (days)',
        'Value': f"{stay_stats['mean']:.1f} ± {stay_stats['std']:.1f}",
        'N': df['LENGTH_OF_STAY'].notna().sum()
    })
    
    # Length of stay by top 5 purposes
    print("Analyzing length of stay by purpose of visit...")
    top_5_purposes = df['PURPOSE OF VISIT'].value_counts().head(5).index
    results.append({
        'Category': 'Travel',
        'Characteristic': 'Length of Stay by Purpose (Top 5)',
        'Value': '',
        'N': '',
        'Percentage': ''
    })
    
    for purpose in top_5_purposes:
        purpose_stats = df[df['PURPOSE OF VISIT'] == purpose]['LENGTH_OF_STAY'].agg(['mean', 'std', 'size'])
        results.append({
            'Category': 'Travel',
            'Characteristic': f"   {purpose}",
            'Value': f"{purpose_stats['mean']:.1f} ± {purpose_stats['std']:.1f}",
            'N': purpose_stats['size']
        })
    
    return pd.DataFrame(results)

def show_message(title, message, error=False):
    """Show message dialog"""
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    try:
        if error:
            messagebox.showerror(title, message)
        else:
            messagebox.showinfo(title, message)
    finally:
        root.destroy()

def main():
    try:
        # File type definitions
        excel_types = [
            ('Excel files', '*.xlsx *.xls*.csv'),
            ('All files', '*.*')
        ]
        
        # Get input file
        input_file = select_file(
            title='Select Excel Data File',
            file_types=excel_types
        )
        
        if not input_file:
            print("No file selected. Exiting.")
            return
        
        # Read the data
        print("Reading data...")
        df = pd.read_excel(
            input_file,
            usecols=['DOB', 'GENDER', 'MARITAL STATUS', 'NATIONALITY',
                    'PURPOSE OF VISIT', 'CARRIER TYPE', 'CARRIER NAME', 
                    'EMBARK PORT', 'TRAVEL DATE', 'RETURN DATE']
        )
        
        # Optimize DataFrame
        print("Optimizing data structure...")
        df = optimize_dataframe(df)
        
        # Create Table 1
        print("Generating Table 1...")
        table1_df = create_table1(df)
        
        # Get output file
        save_types = [
            ('Excel files', '*.xlsx'),
            ('CSV files', '*.csv'),
            ('All files', '*.*')
        ]
        
        output_file = select_file(
            title='Save Table 1 As',
            file_types=save_types,
            save=True
        )
        
        if not output_file:
            print("No output location selected. Exiting.")
            return
        
        # Ensure proper file extension
        if not output_file.endswith(('.xlsx', '.csv')):
            output_file += '.xlsx'
        
        # Save based on extension
        print("Saving results...")
        if output_file.endswith('.xlsx'):
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                table1_df.to_excel(writer, sheet_name='Table 1', index=False)
                
                # Auto-adjust column widths
                worksheet = writer.sheets['Table 1']
                for idx, col in enumerate(table1_df.columns):
                    max_length = max(
                        table1_df[col].astype(str).apply(len).max(),
                        len(col)
                    )
                    worksheet.column_dimensions[chr(65 + idx)].width = max_length + 2
        else:
            table1_df.to_csv(output_file, index=False)
        
        print(f"Table 1 has been saved to: {output_file}")
        show_message("Success", f"Table 1 has been successfully created and saved to:\n{output_file}")
        
    except Exception as e:
        error_msg = f"An error occurred:\n{str(e)}"
        print(error_msg)
        show_message("Error", error_msg, error=True)

if __name__ == "__main__":
    main()

*27/01/2026
Inclusion of descriptive statistics for immigrant density. 


In [ ]:
import os

# Set the desired directory
os.chdir('/Users/janai/Documents/Documents – Janai’s MacBook Air/Personal Files/Learnings/GitHub/Tourism-Research-Paper')

# Verify the change
print(os.getcwd())

Attempt 1

In [ ]:
import pandas as pd
import numpy as np
import os

# Create outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Load the cleaned data
df = pd.read_csv('outputs/df_final.csv')

# Exponentiate the logged immigrant population to get actual values
df['immigrant_population_actual'] = np.exp(df['immigrant_population_log'])

# Get average immigrant density and population by state
state_immigrant_stats = df.groupby('us_state_enc').agg({
    'immigrant_density': 'mean',
    'immigrant_population_actual': 'mean'
}).reset_index()

# Identify top 5 states by immigrant density
top5_states = state_immigrant_stats.nlargest(5, 'immigrant_density')['us_state_enc'].tolist()

print("Top 5 States by Immigrant Density:")
print(top5_states)
print("\n" + "="*80 + "\n")

# Filter data for top 5 states
df_top5 = df[df['us_state_enc'].isin(top5_states)]

# Calculate summary statistics for immigrant density
print("IMMIGRANT DENSITY - Top 5 States Summary Statistics")
print("="*80)
immigrant_density_stats = df_top5.groupby('us_state_enc')['immigrant_density'].describe()
print(immigrant_density_stats)
print("\n")

# Calculate overall statistics for top 5 states
print("Overall Statistics (Top 5 States Combined):")
print(f"Mean: {df_top5['immigrant_density'].mean():.4f}")
print(f"Std: {df_top5['immigrant_density'].std():.4f}")
print(f"Min: {df_top5['immigrant_density'].min():.4f}")
print(f"Max: {df_top5['immigrant_density'].max():.4f}")
print("\n" + "="*80 + "\n")

# Calculate summary statistics for immigrant population (exponentiated)
print("IMMIGRANT POPULATION - Top 5 States Summary Statistics")
print("="*80)
immigrant_pop_stats = df_top5.groupby('us_state_enc')['immigrant_population_actual'].describe()
print(immigrant_pop_stats)
print("\n")

# Calculate overall statistics for top 5 states
print("Overall Statistics (Top 5 States Combined):")
print(f"Mean: {df_top5['immigrant_population_actual'].mean():.0f}")
print(f"Std: {df_top5['immigrant_population_actual'].std():.0f}")
print(f"Min: {df_top5['immigrant_population_actual'].min():.0f}")
print(f"Max: {df_top5['immigrant_population_actual'].max():.0f}")
print("\n" + "="*80 + "\n")

# Create a summary table
summary_table = pd.DataFrame({
    'State': top5_states
})

# Add immigrant density statistics
for state in top5_states:
    state_data = df_top5[df_top5['us_state_enc'] == state]
    density_mean = state_data['immigrant_density'].mean()
    density_std = state_data['immigrant_density'].std()
    pop_mean = state_data['immigrant_population_actual'].mean()
    pop_std = state_data['immigrant_population_actual'].std()
    
    summary_table.loc[summary_table['State'] == state, 'Immigrant_Density_Mean'] = density_mean
    summary_table.loc[summary_table['State'] == state, 'Immigrant_Density_Std'] = density_std
    summary_table.loc[summary_table['State'] == state, 'Immigrant_Population_Mean'] = pop_mean
    summary_table.loc[summary_table['State'] == state, 'Immigrant_Population_Std'] = pop_std

# Sort by immigrant density
summary_table = summary_table.sort_values('Immigrant_Density_Mean', ascending=False)

print("SUMMARY TABLE - Top 5 States by Immigrant Density")
print("="*80)
print(summary_table.to_string(index=False))
print("\n")

# Save to Excel
output_file = 'outputs/immigrant_stats_top5_states.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Write summary table
    summary_table.to_excel(writer, sheet_name='Summary', index=False)
    
    # Write detailed statistics
    immigrant_density_stats.to_excel(writer, sheet_name='Density_Stats')
    immigrant_pop_stats.to_excel(writer, sheet_name='Population_Stats')
    
    # Format columns
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        for column in worksheet.columns:
            max_length = 0
            column = [cell for cell in column]
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = (max_length + 2)
            worksheet.column_dimensions[column[0].column_letter].width = adjusted_width

print(f"Results saved to: {output_file}")

Atempt 2 with only top 5 showing 

In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned data
df = pd.read_csv('outputs/df_final.csv')

# Exponentiate logged variables to get actual values
df['immigrant_population_actual'] = np.exp(df['immigrant_population_log'])
df['state_percapita_income_actual'] = np.exp(df['state_percapita_income_log'])

# Get average immigrant density by state (using us_state_enc)
state_immigrant_stats = df.groupby('us_state_enc').agg({
    'immigrant_density': 'mean'
}).reset_index()

# Identify top 5 states by immigrant density
top5_states_enc = state_immigrant_stats.nlargest(5, 'immigrant_density')['us_state_enc'].tolist()

print("="*100)
print("TABLE 1: DESCRIPTIVE STATISTICS FOR TOP 5 STATES BY IMMIGRANT DENSITY")
print("="*100)
print(f"\nTop 5 States (encoded): {top5_states_enc}\n")

# Filter data for top 5 states
df_top5 = df[df['us_state_enc'].isin(top5_states_enc)]

# Create results table
results = []

# ============================================================================
# SECTION 1: STATE-LEVEL CHARACTERISTICS
# ============================================================================
results.append(['', 'STATE-LEVEL CHARACTERISTICS', '', '', '', ''])
results.append(['', '', '', '', '', ''])

# 1. Immigrant Density
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['immigrant_density']
    results.append([
        'Immigrant Variables',
        f'  State {state}: Immigrant Density',
        f"{state_data.mean():.4f}",
        f"{state_data.std():.4f}",
        len(state_data),
        ''
    ])

# 2. Immigrant Population
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['immigrant_population_actual']
    results.append([
        'Immigrant Variables',
        f'  State {state}: Immigrant Population',
        f"{state_data.mean():.0f}",
        f"{state_data.std():.0f}",
        len(state_data),
        ''
    ])

# 3. State Unemployment
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['state_unemployment']
    results.append([
        'Economic Indicators',
        f'  State {state}: Unemployment Rate',
        f"{state_data.mean():.4f}",
        f"{state_data.std():.4f}",
        len(state_data),
        ''
    ])

# 4. State Per Capita Income (exponentiated)
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['state_percapita_income_actual']
    results.append([
        'Economic Indicators',
        f'  State {state}: Per Capita Income ($)',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])

# 5. Distance (Economic Distance)
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['economic_distance']
    results.append([
        'Geographic Indicators',
        f'  State {state}: Distance (miles)',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])

# 6. Climate Region
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]
    climate_counts = state_data['climate_distance'].value_counts()
    results.append([
        'Geographic Indicators',
        f'  State {state}: Climate Region (most common)',
        f"{climate_counts.index[0] if len(climate_counts) > 0 else 'N/A'}",
        '',
        len(state_data),
        f"{(climate_counts.iloc[0]/len(state_data)*100):.1f}%" if len(climate_counts) > 0 else 'N/A'
    ])

# ============================================================================
# SECTION 2: LENGTH OF STAY BY STATE
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY (CAPPED) BY STATE', '', '', '', ''])
results.append(['', '', '', '', '', ''])

for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['los_capped']
    results.append([
        'Length of Stay',
        f'  State {state}',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])

# ============================================================================
# SECTION 3: LENGTH OF STAY BY STATE AND ACCOMMODATION TYPE
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY BY STATE AND ACCOMMODATION TYPE', '', '', '', ''])
results.append(['', '', '', '', '', ''])

for state in sorted(top5_states_enc):
    accom_types = df_top5[df_top5['us_state_enc'] == state]['accomd_type_enc'].unique()
    for accom in sorted(accom_types):
        if pd.notna(accom):
            state_accom_data = df_top5[(df_top5['us_state_enc'] == state) & 
                                       (df_top5['accomd_type_enc'] == accom)]['los_capped']
            if len(state_accom_data) > 0:
                results.append([
                    'Length of Stay Interactions',
                    f'  State {state} × Accommodation Type {accom}',
                    f"{state_accom_data.mean():.2f}",
                    f"{state_accom_data.std():.2f}",
                    len(state_accom_data),
                    f"{(len(state_accom_data)/len(df_top5)*100):.1f}%"
                ])

# ============================================================================
# SECTION 4: LENGTH OF STAY BY PURPOSE (OVERALL - ALL STATES)
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY BY PURPOSE (ALL TOP 5 STATES)', '', '', '', ''])
results.append(['', '', '', '', '', ''])

purposes = df_top5['purpose_simple'].dropna().unique()
for purpose in sorted(purposes):
    purpose_data = df_top5[df_top5['purpose_simple'] == purpose]['los_capped']
    results.append([
        'Length of Stay by Purpose',
        f'  Purpose {purpose}',
        f"{purpose_data.mean():.2f}",
        f"{purpose_data.std():.2f}",
        len(purpose_data),
        f"{(len(purpose_data)/len(df_top5)*100):.1f}%"
    ])

# ============================================================================
# SECTION 5: AGE BY SEX (OVERALL - ALL STATES)
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'AGE BY SEX (ALL TOP 5 STATES)', '', '', '', ''])
results.append(['', '', '', '', '', ''])

sex_values = df_top5['sex_enc'].dropna().unique()
for sex in sorted(sex_values):
    sex_data = df_top5[df_top5['sex_enc'] == sex]['age']
    sex_label = 'Male' if sex == 1 else 'Female'
    results.append([
        'Demographics',
        f'  {sex_label} (Sex={(sex)})',
        f"{sex_data.mean():.2f}",
        f"{sex_data.std():.2f}",
        len(sex_data),
        f"{(len(sex_data)/len(df_top5)*100):.1f}%"
    ])

# ============================================================================
# CREATE DATAFRAME AND DISPLAY
# ============================================================================
table1_df = pd.DataFrame(results, columns=[
    'Category', 
    'Variable', 
    'Mean', 
    'Std Dev', 
    'N', 
    'Percentage'
])

print(table1_df.to_string(index=False))
print("\n" + "="*100)

# ============================================================================
# SAVE TO EXCEL
# ============================================================================
output_file = 'outputs/Table1_Top5_States_Descriptive_Statistics.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    table1_df.to_excel(writer, sheet_name='Table 1', index=False)
    
    # Format the worksheet
    worksheet = writer.sheets['Table 1']
    
    # Auto-adjust column widths
    for idx, col in enumerate(table1_df.columns):
        max_length = max(
            table1_df[col].astype(str).apply(len).max(),
            len(col)
        )
        worksheet.column_dimensions[chr(65 + idx)].width = min(max_length + 3, 50)
    
    # Make header row bold
    from openpyxl.styles import Font, Alignment
    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')

print(f"\nTable 1 saved to: {output_file}")
print("="*100)

In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned data
df = pd.read_csv('outputs/df_cl.csv')

# Exponentiate logged variables to get actual values
df['immigrant_population_actual'] = np.exp(df['immigrant_population_log'])
df['state_percapita_income_actual'] = np.exp(df['state_percapita_income_log'])

# Get average immigrant density by state (using us_state_enc)
state_immigrant_stats = df.groupby('us_state_enc').agg({
    'immigrant_density': 'mean'
}).reset_index()

# Identify top 5 states by immigrant density
top5_states_enc = state_immigrant_stats.nlargest(5, 'immigrant_density')['us_state_enc'].tolist()

print("="*100)
print("TABLE 1: DESCRIPTIVE STATISTICS FOR TOP 5 STATES + ALL STATES SUMMARY")
print("="*100)
print(f"\nTop 5 States (encoded): {top5_states_enc}\n")

# Filter data for top 5 states
df_top5 = df[df['us_state_enc'].isin(top5_states_enc)]

# Create results table
results = []

# ============================================================================
# SECTION 1: STATE-LEVEL CHARACTERISTICS
# ============================================================================
results.append(['', 'STATE-LEVEL CHARACTERISTICS', '', '', '', ''])
results.append(['', '', '', '', '', ''])

# 1. Immigrant Density - Top 5 + All States
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['immigrant_density']
    results.append([
        'Immigrant Variables',
        f'  State {state}: Immigrant Density',
        f"{state_data.mean():.4f}",
        f"{state_data.std():.4f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['immigrant_density']
results.append([
    'Immigrant Variables',
    '  ALL STATES: Immigrant Density',
    f"{all_states_data.mean():.4f}",
    f"{all_states_data.std():.4f}",
    len(all_states_data),
    ''
])

# 2. Immigrant Population - Top 5 + All States
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['immigrant_population_actual']
    results.append([
        'Immigrant Variables',
        f'  State {state}: Immigrant Population',
        f"{state_data.mean():.0f}",
        f"{state_data.std():.0f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['immigrant_population_actual']
results.append([
    'Immigrant Variables',
    '  ALL STATES: Immigrant Population',
    f"{all_states_data.mean():.0f}",
    f"{all_states_data.std():.0f}",
    len(all_states_data),
    ''
])

# 3. State Unemployment - Top 5 + All States
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['state_unemployment']
    results.append([
        'Economic Indicators',
        f'  State {state}: Unemployment Rate',
        f"{state_data.mean():.4f}",
        f"{state_data.std():.4f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['state_unemployment']
results.append([
    'Economic Indicators',
    '  ALL STATES: Unemployment Rate',
    f"{all_states_data.mean():.4f}",
    f"{all_states_data.std():.4f}",
    len(all_states_data),
    ''
])

# 4. State Per Capita Income (exponentiated) - Top 5 + All States
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['state_percapita_income_actual']
    results.append([
        'Economic Indicators',
        f'  State {state}: Per Capita Income ($)',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['state_percapita_income_actual']
results.append([
    'Economic Indicators',
    '  ALL STATES: Per Capita Income ($)',
    f"{all_states_data.mean():.2f}",
    f"{all_states_data.std():.2f}",
    len(all_states_data),
    ''
])

# 5. Distance (Economic Distance) - Top 5 + All States
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['distance_miles']
    results.append([
        'Geographic Indicators',
        f'  State {state}: Distance (miles)',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['distance_miles']
results.append([
    'Geographic Indicators',
    '  ALL STATES: Distance (miles)',
    f"{all_states_data.mean():.2f}",
    f"{all_states_data.std():.2f}",
    len(all_states_data),
    ''
])

# 6. Climate Region - Top 5 + All States
results.append(['', '', '', '', '', ''])
for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]
    climate_counts = state_data['region_climate'].value_counts()
    results.append([
        'Geographic Indicators',
        f'  State {state}: Climate Region (most common)',
        f"{climate_counts.index[0] if len(climate_counts) > 0 else 'N/A'}",
        '',
        len(state_data),
        f"{(climate_counts.iloc[0]/len(state_data)*100):.1f}%" if len(climate_counts) > 0 else 'N/A'
    ])
# Add all states summary
all_states_climate = df['region_climate'].value_counts()
results.append([
    'Geographic Indicators',
    '  ALL STATES: Climate Region (most common)',
    f"{all_states_climate.index[0] if len(all_states_climate) > 0 else 'N/A'}",
    '',
    len(df),
    f"{(all_states_climate.iloc[0]/len(df)*100):.1f}%" if len(all_states_climate) > 0 else 'N/A'
])

# ============================================================================
# SECTION 2: LENGTH OF STAY BY STATE
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY (CAPPED) BY STATE', '', '', '', ''])
results.append(['', '', '', '', '', ''])

for state in sorted(top5_states_enc):
    state_data = df_top5[df_top5['us_state_enc'] == state]['los_capped']
    results.append([
        'Length of Stay',
        f'  State {state}',
        f"{state_data.mean():.2f}",
        f"{state_data.std():.2f}",
        len(state_data),
        ''
    ])
# Add all states summary
all_states_data = df['los_capped']
results.append([
    'Length of Stay',
    '  ALL STATES',
    f"{all_states_data.mean():.2f}",
    f"{all_states_data.std():.2f}",
    len(all_states_data),
    ''
])

# ============================================================================
# SECTION 3: LENGTH OF STAY BY STATE AND ACCOMMODATION TYPE
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY BY STATE AND ACCOMMODATION TYPE', '', '', '', ''])
results.append(['', '', '', '', '', ''])

for state in sorted(top5_states_enc):
    accom_types = df_top5[df_top5['us_state_enc'] == state]['accomd_type_enc'].unique()
    for accom in sorted(accom_types):
        if pd.notna(accom):
            state_accom_data = df_top5[(df_top5['us_state_enc'] == state) & 
                                       (df_top5['accomd_type_enc'] == accom)]['los_capped']
            if len(state_accom_data) > 0:
                results.append([
                    'Length of Stay Interactions',
                    f'  State {state} × Accommodation Type {int(accom)}',
                    f"{state_accom_data.mean():.2f}",
                    f"{state_accom_data.std():.2f}",
                    len(state_accom_data),
                    f"{(len(state_accom_data)/len(df_top5)*100):.1f}%"
                ])

# Add all states summary by accommodation type
results.append(['', '', '', '', '', ''])
all_accom_types = df['accomd_type_enc'].dropna().unique()
for accom in sorted(all_accom_types):
    all_accom_data = df[df['accomd_type_enc'] == accom]['los_capped']
    results.append([
        'Length of Stay Interactions',
        f'  ALL STATES × Accommodation Type {int(accom)}',
        f"{all_accom_data.mean():.2f}",
        f"{all_accom_data.std():.2f}",
        len(all_accom_data),
        f"{(len(all_accom_data)/len(df)*100):.1f}%"
    ])

# ============================================================================
# SECTION 4: LENGTH OF STAY BY PURPOSE (TOP 5 STATES)
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'LENGTH OF STAY BY PURPOSE (TOP 5 STATES)', '', '', '', ''])
results.append(['', '', '', '', '', ''])

purposes = df_top5['purpose_simple'].dropna().unique()
for purpose in sorted(purposes):
    purpose_data = df_top5[df_top5['purpose_simple'] == purpose]['los_capped']
    results.append([
        'Length of Stay by Purpose',
        f'  Purpose {int(purpose)} (Top 5 States)',
        f"{purpose_data.mean():.2f}",
        f"{purpose_data.std():.2f}",
        len(purpose_data),
        f"{(len(purpose_data)/len(df_top5)*100):.1f}%"
    ])

# Add all states summary by purpose
results.append(['', '', '', '', '', ''])
all_purposes = df['purpose_simple'].dropna().unique()
for purpose in sorted(all_purposes):
    all_purpose_data = df[df['purpose_simple'] == purpose]['los_capped']
    results.append([
        'Length of Stay by Purpose',
        f'  Purpose {int(purpose)} (ALL STATES)',
        f"{all_purpose_data.mean():.2f}",
        f"{all_purpose_data.std():.2f}",
        len(all_purpose_data),
        f"{(len(all_purpose_data)/len(df)*100):.1f}%"
    ])

# ============================================================================
# SECTION 5: AGE BY SEX (TOP 5 STATES)
# ============================================================================
results.append(['', '', '', '', '', ''])
results.append(['', 'AGE BY SEX (TOP 5 STATES)', '', '', '', ''])
results.append(['', '', '', '', '', ''])

sex_values = df_top5['sex_enc'].dropna().unique()
for sex in sorted(sex_values):
    sex_data = df_top5[df_top5['sex_enc'] == sex]['age']
    sex_label = 'Male' if sex == 1 else 'Female'
    results.append([
        'Demographics',
        f'  {sex_label} (Sex={int(sex)}, Top 5 States)',
        f"{sex_data.mean():.2f}",
        f"{sex_data.std():.2f}",
        len(sex_data),
        f"{(len(sex_data)/len(df_top5)*100):.1f}%"
    ])

# Add all states summary by sex
results.append(['', '', '', '', '', ''])
all_sex_values = df['sex_enc'].dropna().unique()
for sex in sorted(all_sex_values):
    all_sex_data = df[df['sex_enc'] == sex]['age']
    sex_label = 'Male' if sex == 1 else 'Female'
    results.append([
        'Demographics',
        f'  {sex_label} (Sex={int(sex)}, ALL STATES)',
        f"{all_sex_data.mean():.2f}",
        f"{all_sex_data.std():.2f}",
        len(all_sex_data),
        f"{(len(all_sex_data)/len(df)*100):.1f}%"
    ])

# ============================================================================
# CREATE DATAFRAME AND DISPLAY
# ============================================================================
table1_df = pd.DataFrame(results, columns=[
    'Category', 
    'Variable', 
    'Mean', 
    'Std Dev', 
    'N', 
    'Percentage'
])

print(table1_df.to_string(index=False))
print("\n" + "="*100)

# ============================================================================
# SAVE TO EXCEL
# ============================================================================
output_file = 'outputs/Table1_Top5_Plus_AllStates_Summary.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    table1_df.to_excel(writer, sheet_name='Table 1', index=False)
    
    # Format the worksheet
    worksheet = writer.sheets['Table 1']
    
    # Auto-adjust column widths
    for idx, col in enumerate(table1_df.columns):
        max_length = max(
            table1_df[col].astype(str).apply(len).max(),
            len(col)
        )
        worksheet.column_dimensions[chr(65 + idx)].width = min(max_length + 3, 50)
    
    # Make header row bold
    from openpyxl.styles import Font, Alignment, PatternFill
    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    # Highlight "ALL STATES" rows with light blue
    blue_fill = PatternFill(start_color='ADD8E6', end_color='ADD8E6', fill_type='solid')
    for row in worksheet.iter_rows(min_row=2, max_row=worksheet.max_row):
        if 'ALL STATES' in str(row[1].value):
            for cell in row:
                cell.fill = blue_fill
                cell.font = Font(bold=True)

print(f"\nTable 1 saved to: {output_file}")
print("\nFormat: Each section shows Top 5 states individually + 1 summary line for ALL STATES")
print("        ALL STATES rows are highlighted in light blue and bold")
print("="*100)